In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

Load Data

In [3]:
X_train  = pd.read_csv('X_train_prepared.csv')
y_train  = pd.read_csv('y_train.csv').squeeze()
X_test   = pd.read_csv('X_test_prepared.csv')
test_ids = pd.read_csv('test_ids.csv').squeeze()

CAT_COLS = ['weekday_of_release', 'season_of_release', 'lunar_phase']
NUM_COLS = [c for c in X_train.columns if c not in CAT_COLS]

print({X_train.shape},{X_test.shape})

{(61609, 54)} {(41074, 54)}


In [5]:
X_tr = X_train.copy()
X_te = X_test.copy()

for c in CAT_COLS:
    X_tr[c] = X_tr[c].fillna('missing').astype(str)
    X_te[c] = X_te[c].fillna('missing').astype(str)

for c in NUM_COLS:
    m = X_tr[c].mean()
    X_tr[c] = X_tr[c].fillna(m)
    X_te[c] = X_te[c].fillna(m)

cat_indices = [X_tr.columns.tolist().index(c) for c in CAT_COLS]
pool_train = Pool(X_tr, y_train, cat_features=cat_indices)
pool_test  = Pool(X_te,          cat_features=cat_indices)

print(f'Categorical indices : {cat_indices}')
print('Pools created.')

Categorical indices : [11, 17, 20]
Pools created.


Train/Test Split + Hyperparameter Tuning

In [14]:
base_params = dict(
    iterations=1200,
    random_seed=42,
    loss_function='RMSE',
    eval_metric='RMSE',
    verbose=0
)

X_trn, X_val, y_trn, y_val = train_test_split(
    X_tr, y_train, test_size=0.3, random_state=42
)

pool_trn = Pool(X_trn, y_trn, cat_features=cat_indices)
pool_val = Pool(X_val, y_val, cat_features=cat_indices)

# Lightweight search: 8 combinations total for faster tuning
param_grid = {
    'learning_rate': [0.03, 0.06],
    'depth': [4, 6],
    'l2_leaf_reg': [3, 8]
}

best_rmse = np.inf
best_params = None
val_model = None

for params in ParameterGrid(param_grid):
    candidate_params = {**base_params, **params}
    candidate = CatBoostRegressor(**candidate_params)
    candidate.fit(
        pool_trn,
        eval_set=pool_val,
        use_best_model=True,
        early_stopping_rounds=80,
        verbose=False
    )
    pred = candidate.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, pred))

    if rmse < best_rmse:
        best_rmse = rmse
        best_params = candidate_params
        val_model = candidate

val_pred = val_model.predict(X_val)
val_rmse = np.sqrt(mean_squared_error(y_val, val_pred))
val_mae = mean_absolute_error(y_val, val_pred)
val_mse = mean_squared_error(y_val, val_pred)
val_r2 = r2_score(y_val, val_pred)


print(f'Best validation RMSE : {val_rmse:.4f}')
print(f'Best params          : {best_params}')
print(f'Best iteration       : {val_model.get_best_iteration()}')
print(f'Validation MAE       : {val_mae:.4f}')
print(f'Validation R²        : {val_r2:.4f}')

Best validation RMSE : 12.9084
Best params          : {'iterations': 1200, 'random_seed': 42, 'loss_function': 'RMSE', 'eval_metric': 'RMSE', 'verbose': 0, 'depth': 6, 'l2_leaf_reg': 3, 'learning_rate': 0.06}
Best iteration       : 1199
Validation MAE       : 9.2603
Validation R²        : 0.6431


## 8. Generate Submission

In [17]:
test_preds = np.clip(candidate.predict(pool_test), 0, 100)
submission = pd.DataFrame({'id': test_ids.values, 'target': test_preds})
submission.to_csv('submission_catboost.csv', index=False)
print('Saved: submission_catboost.csv')

submission.head()

Saved: submission_catboost.csv


,id,target
0,25174,55.040696
1,38453,62.017567
2,29013,57.855035
3,57463,71.408077
4,51264,31.242706


In [19]:
import os

dfs = []
for fname, label in [('split_metrics_linear.csv',  'Linear Regression'),
                     ('split_metrics_xgb.csv',     'XGBoost (Tuned)'),
                     ('split_metrics_catboost.csv','CatBoost')]:
    if os.path.exists(fname):
        row = pd.read_csv(fname)
        row['model'] = label
        dfs.append(row)
    else:
        print(f'Missing: {fname} — run the corresponding notebook first.')

if dfs:
    comparison = pd.concat(dfs, ignore_index=True)[['model', 'split_rmse', 'split_mae', 'split_r2']]
    print('=== Validation Split Metrics Comparison ===')
    print(comparison.to_string(index=False))

=== Validation Split Metrics Comparison ===
            model  split_rmse  split_mae  split_r2
Linear Regression     19.4888    15.7572    0.1865
  XGBoost (Tuned)     11.5345     7.4986    0.7151
         CatBoost     12.9084     9.2603    0.6431
